# Data Source and Preparation

```mermaid
flowchart TB
    subgraph S1[Data Acquisition]
        A[Lazy download unprocessed Mendeley data]
        B[Extract raw MATLAB files]
        C[Select timeSeries .mat files]
    end

    subgraph S2[Signal Preparation]
        D[Load t and 7-column signal matrix d]
        E[Select tool-post accelerometer columns 6-8]
        F[Order-100 Butterworth low-pass filter]
        G[Subsample to 10 kHz]
    end

    subgraph S3[Window Labeling]
        H[Segment into 2.5 s windows]
        I[Compute RMS on X axis]
        J[FFT with Hann window]
        K[Detect harmonic chatter structure]
        L[Assign chatter/no_chatter label]
    end

    subgraph S4[Dataset Output]
        M[Save X/Y/Z windows with fs and metadata as NPZ]
    end

    A --> B --> C --> D --> E --> F --> G --> H
    H --> I --> J --> K --> L --> M
```

The dataset consists of vibration signals recorded during industrial cutting experiments. Each experiment directory corresponds to a specific machining configuration (e.g., tool stick-out length, cutting parameters).

Raw measurements are stored as MATLAB .mat files and contain multi-channel time series data.

In [15]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from scipy.signal import butter, find_peaks, sosfiltfilt

In [16]:
# Download and extract the original dataset from Mendeley (if not already present)
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

NOTEBOOK_REPO_ROOT = Path.cwd().resolve()
if NOTEBOOK_REPO_ROOT.name == "notebooks":
    NOTEBOOK_REPO_ROOT = NOTEBOOK_REPO_ROOT.parent

URL = "https://data.mendeley.com/public-files/datasets/hvm4wh3jzx/files/fe97a01d-0b17-42b4-af73-e5c4f4cf5a56/file_downloaded"
RAW_ZIP = NOTEBOOK_REPO_ROOT / "data" / "raw" / "turning_unprocessed_dataset.zip"
RAW_EXTRACT_DIR = NOTEBOOK_REPO_ROOT / "data" / "raw_turning_unprocessed"
RAW_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if not any(RAW_EXTRACT_DIR.rglob("*.mat")):
    print("Unprocessed turning .mat files not found locally.")

if not RAW_ZIP.exists() and not any(RAW_EXTRACT_DIR.rglob("*.mat")):
    try:
        import requests
        RAW_ZIP.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(URL, stream=True) as r:
            r.raise_for_status()
            with open(RAW_ZIP, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
    except Exception:
        # fallback to urllib
        RAW_ZIP.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(URL, RAW_ZIP)

if RAW_ZIP.exists() and not any(RAW_EXTRACT_DIR.rglob("*.mat")):
    # Try to unzip; if not a zip, keep the downloaded file for inspection
    try:
        with zipfile.ZipFile(RAW_ZIP, 'r') as z:
            z.extractall(RAW_EXTRACT_DIR)
        print('Extracted dataset to', RAW_EXTRACT_DIR)
    except zipfile.BadZipFile:
        print('Downloaded file is not a ZIP archive. Saved to', RAW_ZIP)
else:
    print('Using existing raw data in', RAW_EXTRACT_DIR)


Using existing raw data in /home/cgawron/git/spectrogram-anomaly-ae/data/raw_turning_unprocessed


## Configuration of input/output directory and data
- path to raw data
- output path for processed data
- data definition:
    - Accelerometer axes (x, y, z) are taken from paper columns 6, 7, 8: the tri-axial accelerometer on the tool post
    - USE_CHANNEL: axis for labeling decision (see Turning_Cutting_Data_Description.pdf for labeling approach) --> shows dominant vibration direction in cutting process
- anti-alias preprocessing: raw x/y/z accelerometer channels are low-pass filtered with an order-100 Butterworth filter and then subsampled to 10 kHz
- WINDOW_SECONDS / WINDOW_SAMPLES: to transform the continous timeseries into fixed-size slices, a fixed-duration window is used; the ML-approach is not able to work on continous data, it needs fixed length windows to perform data preparation --> Provides sufficient temporal context for chatter development; 

        - Empirically chosen as trade-off between:
        - temporal resolution (shorter windows)
        - spectral stability (longer windows)


In [17]:
# For debugging select each subfolder separately: 2inchStickout, 2p5inchStickout ...
# Set `INPUT_DIR` to the extracted raw data directory created by the downloader above.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_EXTRACT_DIR = REPO_ROOT / "data" / "raw_turning_unprocessed"
INPUT_DIR = RAW_EXTRACT_DIR  # adjust to a specific experiment subfolder if needed
OUTPUT_BASE = REPO_ROOT / "data" / "01_windowed_labeled_2,5s"

MAT_GLOB = "*.mat"

# Paper columns 6, 7, 8: tri-axial accelerometer on the tool post.
# `d` excludes the time column, so these are zero-based d columns 4, 5, 6.
AXES = ("X", "Y", "Z")
TOOL_POST_ACCEL_D_COLUMNS = [4, 5, 6]
USE_CHANNEL = "X"     # selected axis for labeling data

TARGET_SAMPLE_RATE_HZ = 10_000.0
LOWPASS_FILTER_ORDER = 100
LOWPASS_CUTOFF_HZ = 5_000.0

WINDOW_SECONDS = 2.5
WINDOW_SAMPLES = int(round(WINDOW_SECONDS * TARGET_SAMPLE_RATE_HZ))


# Labeling Strategy

Each time window is evaluated in both the time domain and the frequency domain to determine whether chatter-related vibration is present.

The labeling process combines:

- a time-domain amplitude criterion (vibration intensity)
- a frequency-domain harmonic criterion (chatter signature)

**NOTE**: Chatter is defined purely by harmonic spectral structure, while signal amplitude is used only as auxiliary contextual information

This reflects a practical industrial setting, where threshold-based monitoring systems are the current standard for alarm generation in condition monitoring systems.

A window is classified based on whether a chatter signature is detected:

Chatter present → chatter \
No chatter detected → no_chatter

This approach ensures that the labels remain aligned with industrial monitoring logic, while incorporating additional spectral structure for improved robustness.

## Parameter Selection for Labeling

- F_CHATTER_MAX (Frequency Cutt-Off): 
    - Chatter phenomena in machining typically manifest in low-to-mid frequency range
    - High-frequency components are more affected by sensor noise and structural resonance

- PEAK_AMP_TH (Peak Detection Threshold in Spectrum):
    - filters out low-energy spectral fluctuations and noise-induced peaks
    - Ensures that detected peaks correspond to physically meaningful vibration modes 

- TH_TIME_LOW_MIN / TH_TIME_LOW_MAX :
    - Derived from empirical distribution of vibration amplitudes across experiments
    - Separates low-energy stable cutting from higher-energy regimes

- K_MAX: defines the maximum harmonic order used to evaluate spectral consistency of chatter. It limits the search for harmonic relationships between the fundamental frequency and its integer multiples.

- F0_REL_TOL defines the allowed relative deviation when matching harmonic frequencies, scaling with the harmonic order.
- F0_ABS_TOL ensures a minimum absolute tolerance in Hz, preventing overly strict matching at low frequencies.


In [18]:
F_CHATTER_MAX = TARGET_SAMPLE_RATE_HZ / 2  # below 5 kHz after subsampling to 10 kHz
PEAK_AMP_TH = 0.001        # treshold for spectrum 0.00055

# threshold for time representation (rms)
TH_TIME_LOW_MIN = 0.02
TH_TIME_LOW_MAX = 0.06

# Harmonic-detection
K_MAX = 10                   
F0_REL_TOL = 0.02           # percentual tolerance (±2%)
F0_ABS_TOL = 5.0             # add. abs. tolerance [Hz]

# FFT window function
USE_WINDOW_FUNCTION = True  # Hann window

## Output Organisation
- extract experiment names for annotate files

In [19]:
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

mat_files = sorted(INPUT_DIR.glob(f"timeSeries_*/{MAT_GLOB}"))
print("Found mat files:", len(mat_files), "input:", INPUT_DIR)

Found mat files: 64 input: /home/cgawron/git/spectrogram-anomaly-ae/data/raw_turning_unprocessed


# Signal Processing

- **peak_abs:** Measures maximum absolute vibration amplitude within a time window
- **compute_rfft_amp:** Converts time-domain vibration signals into normalized frequency-domain amplitude spectra for harmonic analysis
- **has_harmonic_chatter:** Chatter is detected as a signal exhibiting a stable harmonic ladder structure in the frequency dominant
- **segment_time_indices:** Converts continuous vibration measurements into fixed-length segments suitable for machine learning and spectrogram generation

In [20]:
def load_raw_mat_timeseries(mat_file: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load raw time vector and signal matrix from supported MATLAB files."""
    mat = loadmat(mat_file)

    if "t" in mat and "d" in mat:
        t = np.asarray(mat["t"]).reshape(-1)
        d = np.asarray(mat["d"])
    elif "tsDS" in mat:
        ts = np.asarray(mat["tsDS"])
        if ts.ndim != 2 or ts.shape[1] < 2:
            raise ValueError(f"{mat_file} has unsupported tsDS shape {ts.shape}")
        t = ts[:, 0]
        d = ts[:, 1:]
    else:
        keys = sorted(k for k in mat if not k.startswith("__"))
        raise ValueError(f"{mat_file} does not contain 't'/'d' or 'tsDS'. Available keys: {keys}")

    if d.ndim != 2:
        raise ValueError(f"{mat_file} signal matrix must be 2-D, got shape {d.shape}")
    if len(t) != len(d):
        raise ValueError(f"{mat_file} time and signal lengths differ: {len(t)} vs {len(d)}")
    return t.astype(float), d.astype(float)


def estimate_sample_rate_hz(t: np.ndarray) -> float:
    if len(t) < 2:
        raise ValueError("Time vector must contain at least two samples")
    dt = np.median(np.diff(t))
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("Invalid or non-increasing time vector")
    return float(1.0 / dt)


def select_tool_post_axes(d: np.ndarray, mat_file: Path) -> np.ndarray:
    """Return X/Y/Z from paper columns 6-8, represented as d columns 4-6."""
    max_column = max(TOOL_POST_ACCEL_D_COLUMNS)
    if d.shape[1] <= max_column:
        raise ValueError(
            f"{mat_file} has {d.shape[1]} signal columns, but tool-post X/Y/Z need "
            f"d columns {TOOL_POST_ACCEL_D_COLUMNS}. Use the original multi-channel raw files."
        )
    return d[:, TOOL_POST_ACCEL_D_COLUMNS]


def output_relative_parent(mat_file: Path) -> Path:
    """Preserve raw subfolders while matching the existing stickout folder style."""
    parts = list(mat_file.parent.relative_to(INPUT_DIR).parts)
    if parts:
        parts[0] = parts[0].replace("timeSeries_", "", 1).replace("_stickout", "Stickout")
    return Path(*parts) if parts else Path()


def lowpass_filter_and_subsample(
    t: np.ndarray,
    xyz: np.ndarray,
    fs_raw: float,
    target_fs: float = TARGET_SAMPLE_RATE_HZ,
    cutoff_hz: float = LOWPASS_CUTOFF_HZ,
    filter_order: int = LOWPASS_FILTER_ORDER,
) -> tuple[np.ndarray, np.ndarray, float, int]:
    """Butterworth low-pass filter and integer-decimate to the target sample rate."""
    fs_raw = float(fs_raw)
    target_fs = float(target_fs)

    if fs_raw < target_fs and not np.isclose(fs_raw, target_fs, rtol=1e-4, atol=1e-6):
        raise ValueError(f"Raw sample rate {fs_raw:.3f} Hz is below target {target_fs:.3f} Hz")

    if np.isclose(fs_raw, target_fs, rtol=1e-4, atol=1e-6):
        return t, xyz, fs_raw, 1

    subsample_factor = int(round(fs_raw / target_fs))
    fs = fs_raw / subsample_factor
    if not np.isclose(fs, target_fs, rtol=1e-4, atol=1e-6):
        raise ValueError(
            f"Raw sample rate {fs_raw:.3f} Hz is not an integer multiple of target "
            f"{target_fs:.3f} Hz"
        )

    if cutoff_hz >= fs_raw / 2:
        raise ValueError(f"Low-pass cutoff {cutoff_hz} Hz must be below raw Nyquist {fs_raw / 2} Hz")

    sos = butter(filter_order, cutoff_hz, btype="lowpass", fs=fs_raw, output="sos")
    xyz_filtered = sosfiltfilt(sos, xyz, axis=0)
    xyz_subsampled = xyz_filtered[::subsample_factor]
    t_subsampled = t[0] + np.arange(len(xyz_subsampled)) / fs
    return t_subsampled, xyz_subsampled, float(fs), subsample_factor


def peak_abs(x: np.ndarray) -> float:
    return float(np.max(np.abs(x)))

def compute_rfft_amp(x: np.ndarray, fs: float, use_window=True):
    N = len(x)
    if N < 2:
        raise ValueError("Segment zu kurz für FFT.")

    if use_window:
        w = np.hanning(N)
        xw = x * w
        # einfache Skalierungskorrektur für Amplituden
        win_correction = np.sum(w) / N
        win_correction = max(win_correction, 1e-12)
    else:
        xw = x
        win_correction = 1.0

    freqs = np.fft.rfftfreq(N, d=1/fs)
    fft = np.fft.rfft(xw)
    amp = np.abs(fft) / N / win_correction
    return freqs, amp

def has_harmonic_chatter(freqs, amp,
                          f_max=5000.0,
                          peak_amp_th=0.00025,
                          k_max=10,
                          f0_rel_tol=0.02,
                          f0_abs_tol=5.0):
    """
    Heuristik:
    - finde Peaks (lokale Maxima) in [0, f_max] mit amp > peak_amp_th
    - bestimme f0 als Frequenz des stärksten Peaks
    - prüfe, ob viele Harmonische bei k*f0 existieren (mit Toleranz)
    """
    # Bereich einschränken
    mask = (freqs > 0) & (freqs <= f_max)
    freqs_r = freqs[mask]
    amp_r = amp[mask]

    if len(freqs_r) == 0:
        return False, {"n_peaks": 0, "f0": np.nan, "n_harmonics_found": 0}

    # Peaks finden
    peaks, props = find_peaks(amp_r, height=peak_amp_th)
    if len(peaks) == 0:
        return False, {"n_peaks": 0, "f0": np.nan, "n_harmonics_found": 0}

    peak_freqs = freqs_r[peaks]
    peak_amps = props["peak_heights"]

    # f0 = stärkster Peak
    idx_max = int(np.argmax(peak_amps))
    f0 = float(peak_freqs[idx_max])

    # vorhandene Peak-Frequenzen (für schnelle Checks)
    peak_set = peak_freqs  # numpy array

    # prüfe Harmonische
    n_found = 0
    harmonics = []
    for k in range(2, k_max + 1):
        target = k * f0
        if target > f_max:
            break

        tol = max(f0_rel_tol * target, f0_abs_tol)
        # finde Peak, der im Toleranzband liegt
        if np.any(np.abs(peak_set - target) <= tol):
            n_found += 1
            harmonics.append(k)

    # Kriterium: mind. z.B. 2 Harmonische über Threshold -> "deutliches Rattern"
    # (kannst du später leicht anpassen)
    has = n_found >= 3

    return has, {"n_peaks": int(len(peaks)), "f0": f0, "n_harmonics_found": int(n_found), "harmonics": harmonics}

def segment_time_indices(t, window_samples=WINDOW_SAMPLES):
    win_len = int(window_samples)
    if win_len <= 0:
        raise ValueError("window_samples must be positive")
    if len(t) == 0:
        return
    if len(t) < win_len:
        yield 0, 0, len(t)
        return

    n_windows = len(t) // win_len
    for w in range(n_windows):
        s = w * win_len
        e = s + win_len
        yield w, s, e

In [21]:
ch_idx = AXES.index(USE_CHANNEL)

all_manifest = []

for mat_file in mat_files:
    print("Processing:", mat_file.name)

    try:
        t_raw, d_raw = load_raw_mat_timeseries(mat_file)
        fs_raw = estimate_sample_rate_hz(t_raw)
        xyz_raw = select_tool_post_axes(d_raw, mat_file)
        t, xyz, fs, subsample_factor = lowpass_filter_and_subsample(t_raw, xyz_raw, fs_raw)
    except ValueError as exc:
        print(f"  skipping: {exc}")
        continue

    # Referenzsignal für Labeling (X-Achse)
    x_label = xyz[:, ch_idx]

    segments = list(segment_time_indices(t, window_samples=WINDOW_SAMPLES))
    print("  segments:", len(segments))

    relative_parent = output_relative_parent(mat_file)
    out_dir_mat = OUTPUT_BASE / relative_parent / mat_file.stem
    out_dir_mat.mkdir(parents=True, exist_ok=True)

    rows = []

    for w, s, e in segments:
        t_seg = t[s:e]
        x_seg = x_label[s:e]
        actual_window_samples = e - s

        # =========================
        # Zeitdomäne (RMS korrekt)
        # =========================
        A_time = np.sqrt(np.mean(x_seg ** 2))

        # =========================
        # Frequenzdomäne (Labeling)
        # =========================
        freqs, amp = compute_rfft_amp(
            x_seg,
            fs,
            use_window=USE_WINDOW_FUNCTION
        )

        #mask = freqs <= F_CHATTER_MAX
        #freqs_f = freqs[mask]
        #amp_f = amp[mask]

        is_chatter, freq_info = has_harmonic_chatter(
            freqs,
            amp,
            f_max=F_CHATTER_MAX,
            peak_amp_th=PEAK_AMP_TH,
            k_max=K_MAX
        )
        # =========================
        # Labeling
        # =========================

        if is_chatter:
            label = "chatter"
        else:
            label = "no_chatter"

        # =========================
        # Speichern NPZ (3-Achsen!)
        # =========================
        seg_path = out_dir_mat / f"{mat_file.stem}_window_{w:04d}.npz"

        np.savez_compressed(
            seg_path,
            t=t_seg,
            X=xyz[s:e, 0],
            Y=xyz[s:e, 1],
            Z=xyz[s:e, 2],
            label=label,
            A_time=A_time,
            is_chatter=is_chatter,
            fs=float(fs),
            fs_raw=float(fs_raw),
            lowpass_filter_order=LOWPASS_FILTER_ORDER,
            lowpass_cutoff_hz=LOWPASS_CUTOFF_HZ,
            subsample_factor=subsample_factor,
            source_d_columns=np.array(TOOL_POST_ACCEL_D_COLUMNS, dtype=int),
            target_window_samples=WINDOW_SAMPLES,
            window_samples=actual_window_samples,
            window_seconds=actual_window_samples / fs
        )

        # =========================
        # Manifest
        # =========================
        rows.append({
            "window": w,
            "label": label,
            "is_chatter": is_chatter,
            "A_time": A_time,
            "fs": fs,
            "fs_raw": fs_raw,
            "lowpass_filter_order": LOWPASS_FILTER_ORDER,
            "lowpass_cutoff_hz": LOWPASS_CUTOFF_HZ,
            "subsample_factor": subsample_factor,
            "source_d_columns": ",".join(map(str, TOOL_POST_ACCEL_D_COLUMNS)),
            "target_window_samples": WINDOW_SAMPLES,
            "window_samples": actual_window_samples,
            "window_seconds": actual_window_samples / fs,
            "segment_path": str(seg_path)
        })

    pd.DataFrame(rows).to_csv(
        out_dir_mat / f"{out_dir_mat.parent.name}_label.csv",
        index=False
    )

    all_manifest.append(pd.DataFrame(rows))

print("DONE. Saved:", OUTPUT_BASE)

Processing: F_08-Jun-2017_rpm1030_doc0p001.mat
  segments: 4
Processing: F_08-Jun-2017_rpm320_doc0p005.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p010.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p015.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p020.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p025.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p030.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p035.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p040.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p045.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p050.mat
  segments: 24
Processing: F_08-Jun-2017_rpm320_doc0p060_xAccDetached.mat
  segments: 12
Processing: F_08-Jun-2017_rpm425_doc0p005.mat
  segments: 8
Processing: F_08-Jun-2017_rpm425_doc0p010.mat
  segments: 8
Processing: F_08-Jun-2017_rpm425_doc0p015.mat
  segments: 8
Processing: F_08-Jun-2017_rpm425_doc0p017.mat
  segments: 8
Processing: F_0